# RISE：主成分得分来自图像的哪个位置

随机遮挡图像 → 重新提特征 → 投影到已拟合的主成分 → 由得分的变化得到显著图（红 = 该区域把得分推高，蓝 = 推低）。
输出 `results/rise/<model>/<设定>/<image_id>.npy`（`[K, H, W]`）和 `figures/rise_<model>.png`。
先运行初始化格；其余各格只依赖它，顺序随意。路径在 `fmspace/paths.py`。

In [ ]:
import time

import numpy as np
import pandas as pd
import torch
from PIL import Image

from fmspace.analysis import l2, pc_scorer, pca, rho
from fmspace.data import load_metadata
from fmspace.extract import load_features
from fmspace.models import load_encoder
from fmspace.paths import DATA_DIR, FEATURE_DIR, FIGURE_DIR, RISE_DIR
from fmspace.plots import saliency_figure
from fmspace.rise import blurred, fov_mask, load_image, rise

MODEL = "medsiglip"
COMPONENTS = {1: "eye_dr_level", 2: "eye"}  # 主成分 → 定方向用的标签（得分调成与它正相关；eye 为右眼正）
N_MASKS, GRID, SEED = 1000, 8, 0  # 试跑用 1000；正式跑 4000 以上
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

metadata = load_metadata(DATA_DIR).set_index("image_id", drop=False)
X = l2(load_features(FEATURE_DIR / f"{MODEL}.npz", metadata["image_id"]))
fitted = pca(X)  # 与 analysis.ipynb 相同：全部图像拟合，RISE 中不重新拟合
pcs = fitted.transform(X)[:, np.array(list(COMPONENTS)) - 1]
signs = np.array([np.sign(rho(pcs[:, j], metadata[label].rank())) for j, label in enumerate(COMPONENTS.values())])
scores = pd.DataFrame(pcs * signs, index=metadata.index, columns=[f"PC{k}" for k in COMPONENTS])

encode = load_encoder(MODEL, DEVICE)
score = pc_scorer(encode, fitted, list(COMPONENTS), signs)
OUT = RISE_DIR / MODEL / f"pc{'-'.join(map(str, COMPONENTS))}_grid{GRID}_n{N_MASKS}"
OUT.mkdir(parents=True, exist_ok=True)

## 复现检查

重新提几张图的特征，和 `features/<model>.npz` 逐张比余弦相似度。`full` 用原图，`resized` 用缩到 512 px 的图（RISE 用的尺寸）；两列都应接近 1。

In [ ]:
ids = metadata["image_id"][:8]
with torch.inference_mode():
    z = {
        "full": encode([Image.open(metadata["path"][i]).convert("RGB") for i in ids]),
        "resized": encode([load_image(metadata["path"][i]) for i in ids]),
    }
pd.DataFrame({k: (l2(v.float().cpu().numpy()) * X[metadata.index.get_indexer(ids)]).sum(axis=1) for k, v in z.items()}, index=ids)

## 计时

In [ ]:
images = [load_image(metadata["path"].iloc[0])] * 8
start = time.perf_counter()
score(images)
seconds = (time.perf_counter() - start) / len(images)
print(f"{DEVICE}：{seconds:.2f} 秒/次前向，{N_MASKS} 张掩膜约 {seconds * N_MASKS / 60:.0f} 分钟/图")

## 跑 RISE

按 `BY` 的得分取最低、最高各 `N` 张（每名患者最多 1 张）。已有结果的图跳过。

In [ ]:
BY, N = "PC1", 1

ranked = scores.join(metadata[["patient_id", "eye", "eye_dr_level"]]).sort_values(BY)
selected = pd.concat([ranked.drop_duplicates("patient_id").head(N), ranked.drop_duplicates("patient_id", keep="last").tail(N)])
display(selected)

for image_id in selected.index:
    if not (OUT / f"{image_id}.npy").exists():
        image = load_image(metadata["path"][image_id])
        np.save(OUT / f"{image_id}.npy", rise(score, image, blurred(image), N_MASKS, GRID, seed=SEED))

## 画图

画 `OUT` 里已有的全部结果，按第一个主成分的得分排序。

In [ ]:
ids = scores.loc[[p.stem for p in OUT.glob("*.npy")]].sort_values(scores.columns[0]).index
images = [load_image(metadata["path"][i]) for i in ids]
labels = [f"DR {metadata['eye_dr_level'][i]}, {metadata['eye'][i]}\n" + ", ".join(f"{c} {scores[c][i]:+.2f}" for c in scores) for i in ids]
fig = saliency_figure(images, [np.load(OUT / f"{i}.npy") for i in ids], [fov_mask(im) for im in images], labels, list(scores.columns))
fig.savefig(FIGURE_DIR / f"rise_{MODEL}.png", dpi=300, bbox_inches="tight")

## 检查：掩膜数够不够、填充的影响

同一张图（第一个主成分得分最高的图）跑三次：随机种子 0、随机种子 1、黑色填充。视野圆内两张显著图的 Spearman：
`seed` 接近 1 说明掩膜数够；`fill` 低说明结果依赖填充方式。

In [ ]:
image = load_image(metadata["path"][scores.iloc[:, 0].idxmax()])
fov = fov_mask(image)
a = rise(score, image, blurred(image), N_MASKS, GRID, seed=0)
b = rise(score, image, blurred(image), N_MASKS, GRID, seed=1)
c = rise(score, image, Image.new("RGB", image.size), N_MASKS, GRID, seed=0)
pd.DataFrame({"seed": [rho(x[fov], y[fov]) for x, y in zip(a, b)], "fill": [rho(x[fov], y[fov]) for x, y in zip(a, c)]}, index=scores.columns)